In [1]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'  # Force CPU to avoid BLAS issues

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from typing import Callable
import optax

import pcx as px
import pcx.predictive_coding as pxc
import pcx.nn as pxnn
import pcx.functional as pxf
import pcx.utils as pxu

px.RKG.seed(42)

ModuleNotFoundError: No module named 'jax'

In [2]:
class DEQ_PC(pxc.EnergyModule):
    """Deep Equilibrium Predictive Coding Network.
    
    Architecture (z is shared first/last):
        z.h + embed(x) → layer1 → act → v_mid → layer2 → z.u
        z.h → readout → v_out (classification)
    
    At equilibrium: z.h = z.u = layer2(act(layer1(z.h + embed(x)))) = f(z.h, x)
    This IS the DEQ fixed-point condition!
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, act_fn: Callable):
        super().__init__()
        
        self.act_fn = px.static(act_fn)
        self.hidden_dim = px.static(hidden_dim)
        
        # Input embedding
        self.embed = pxnn.Linear(input_dim, hidden_dim)
        
        # DEQ recurrent block: f(z, x) = layer2(act(layer1(z + embed(x))))
        self.layer1 = pxnn.Linear(hidden_dim, hidden_dim)
        self.layer2 = pxnn.Linear(hidden_dim, hidden_dim)
        
        # Classification readout
        self.readout = pxnn.Linear(hidden_dim, output_dim)
        
        # The SHARED fixed-point vode (first = last)
        self.z = pxc.Vode(energy_fn=pxc.se_energy)
        
        # Intermediate vode
        self.v_mid = pxc.Vode(energy_fn=pxc.se_energy)
        
        # Classification output
        self.v_out = pxc.Vode(energy_fn=pxc.ce_energy)
        self.v_out.h.frozen = True
    
    def __call__(self, x, y=None):
        z_h = self.z.get("h")
        
        # DEQ loop: z.h → (+embed) → layer1 → act → v_mid → layer2 → z.u
        combined = z_h + self.embed(x)
        self.v_mid(self.act_fn(self.layer1(combined)))
        self.z(self.layer2(self.v_mid.get("h")))
        
        # Classification branch
        self.v_out(self.readout(z_h))
        if y is not None:
            self.v_out.set("h", y)
        
        return self.v_out.get("u")

NameError: name 'pxc' is not defined

In [3]:
# Batched operations
@pxf.vmap(pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)), in_axes=(0, 0), out_axes=0)
def forward(x, y, *, model):
    return model(x, y)

@pxf.vmap(pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)), in_axes=(0,), out_axes=(None, 0), axis_name="batch")
def energy(x, *, model):
    y_pred = model(x, None)
    return jax.lax.psum(model.energy(), "batch"), y_pred

# DEQ-specific training: z.h initialized randomly (not via forward pass)
@pxf.jit(static_argnums=0)
def train_deq_batch(T, x, y, *, model, optim_h, optim_w):
    model.train()
    batch_size = x.shape[0]
    hidden_dim = model.hidden_dim.get()
    
    # KEY DIFFERENCE: Initialize z.h uniformly at random (DEQ style)
    # This is the "initial guess" for the fixed point
    z_init = jax.random.uniform(px.RKG(), (batch_size, hidden_dim), minval=-0.1, maxval=0.1)
    model.z.set("h", z_init)
    
    # Forward pass to initialize other vodes
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        forward(x, y, model=model)
    
    # Restore z.h to random (STATUS.INIT set h=u, we want random start)
    model.z.set("h", z_init)
    
    optim_h.init(pxu.M_hasnot(pxc.VodeParam, frozen=True)(model))
    
    # Inference: find the fixed point via energy minimization
    for _ in range(T):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (e, _), g = pxf.value_and_grad(
                pxu.M_hasnot(pxc.VodeParam, frozen=True).to([False, True]), has_aux=True
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
    
    optim_h.clear()
    
    # Weight update at equilibrium
    with pxu.step(model, clear_params=pxc.VodeParam.Cache):
        (e, _), g = pxf.value_and_grad(pxu.M(pxnn.LayerParam).to([False, True]), has_aux=True)(energy)(x, model=model)
    optim_w.step(model, g["model"], scale_by=1.0/x.shape[0])
    
    return e

NameError: name 'pxf' is not defined

In [4]:
# Training with energy history for analysis (non-JIT version for flexibility)
def train_deq_with_history(T, x, y, *, model, optim_h, optim_w):
    model.train()
    batch_size = x.shape[0]
    hidden_dim = model.hidden_dim.get()
    
    z_init = jax.random.uniform(px.RKG(), (batch_size, hidden_dim), minval=-0.1, maxval=0.1)
    model.z.set("h", z_init)
    
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        forward(x, y, model=model)
    model.z.set("h", z_init)
    
    optim_h.init(pxu.M_hasnot(pxc.VodeParam, frozen=True)(model))
    
    # Collect energies - without JIT this is more flexible
    energies = []
    for _ in range(T):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (e, _), g = pxf.value_and_grad(
                pxu.M_hasnot(pxc.VodeParam, frozen=True).to([False, True]), has_aux=True
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
        energies.append(float(e))  # Convert to Python float to avoid array accumulation
    
    optim_h.clear()
    
    with pxu.step(model, clear_params=pxc.VodeParam.Cache):
        (e, _), g = pxf.value_and_grad(pxu.M(pxnn.LayerParam).to([False, True]), has_aux=True)(energy)(x, model=model)
    optim_w.step(model, g["model"], scale_by=1.0/x.shape[0])
    
    return jnp.array(energies)

In [5]:
# FIXED EVAL: Must run inference to find fixed point!
@pxf.jit(static_argnums=0)
def eval_deq_batch(T, x, y, *, model, optim_h):
    model.train()  # Need train mode for inference dynamics
    batch_size = x.shape[0]
    hidden_dim = model.hidden_dim.get()
    
    # Initialize z.h to zeros
    z_init = jnp.zeros((batch_size, hidden_dim))
    model.z.set("h", z_init)
    
    # Initialize v_mid via forward pass (y=None means no label supervision)
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        forward(x, None, model=model)
    
    # Restore z.h to zeros
    model.z.set("h", z_init)
    
    # KEY FIX: Neutralize v_out.energy by setting v_out.h = v_out.u
    # This way inference only minimizes the DEQ fixed-point energy
    model.v_out.h.frozen = False  # Temporarily unfreeze
    
    optim_h.init(pxu.M(pxc.VodeParam)(model))  # All vodes including v_out
    
    for _ in range(T):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            # Forward to compute u values
            forward(x, None, model=model)
            # Set v_out.h = v_out.u to zero out classification energy
            model.v_out.set("h", model.v_out.get("u"))
        
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (e, _), g = pxf.value_and_grad(
                pxu.M(pxc.VodeParam).to([False, True]), has_aux=True
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
    
    optim_h.clear()
    model.v_out.h.frozen = True  # Re-freeze
    
    # Final forward with converged z.h to get prediction
    with pxu.step(model, clear_params=pxc.VodeParam.Cache):
        y_pred = forward(x, None, model=model).argmax(axis=-1)
    
    return (y_pred == y).sum()

NameError: name 'pxf' is not defined

In [6]:
# Load MNIST
print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist = mnist.data.astype(np.float32) / 255.0
y_mnist = mnist.target.astype(np.int32)

X_train, y_train = X_mnist[:60000], y_mnist[:60000]
X_test, y_test = X_mnist[60000:], y_mnist[60000:]

batch_size = 256
n_train = (len(X_train) // batch_size) * batch_size
n_test = (len(X_test) // batch_size) * batch_size

X_train_batches = X_train[:n_train].reshape(-1, batch_size, 784)
y_train_batches = y_train[:n_train].reshape(-1, batch_size)
X_test_batches = X_test[:n_test].reshape(-1, batch_size, 784)
y_test_batches = y_test[:n_test].reshape(-1, batch_size)

print(f"Train batches: {len(X_train_batches)}, Test batches: {len(X_test_batches)}")

# New Cell
optimizer_configs = {
    "SGD (Euler)": lambda: optax.sgd(5e-2),
    "Momentum (Heavy Ball)": lambda: optax.sgd(5e-2, momentum=0.9),
    #"Nesterov (Symplectic)": lambda: optax.sgd(5e-2, momentum=0.9, nesterov=True),
    #"Damped Harmonic": lambda: optax.sgd(5e-2, momentum=0.95),
}

Loading MNIST...


NameError: name 'fetch_openml' is not defined

In [7]:
n_epochs = 10
T_inference = 100

deq_results = {}

for name, optim_fn in optimizer_configs.items():
    print(f"\n{'='*50}")
    print(f"DEQ-PC with {name}")
    print('='*50)
    
    px.RKG.seed(42)
    model = DEQ_PC(input_dim=784, hidden_dim=32, output_dim=10, act_fn=jax.nn.relu)
    
    # Initialize model
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        model.z.set("h", jnp.zeros((batch_size, 32)))
        forward(jnp.zeros((batch_size, 784)), None, model=model)
    
    optim_h = pxu.Optim(optim_fn)
    optim_w = pxu.Optim(lambda: optax.adamw(1e-3), pxu.M(pxnn.LayerParam)(model))
    
    first_batch_energies = []
    
    for epoch in range(n_epochs):
        energies = train_deq_with_history(
            T_inference,
            X_train_batches[0],
            jax.nn.one_hot(y_train_batches[0], 10),
            model=model, optim_h=optim_h, optim_w=optim_w
        )
        first_batch_energies.append(np.array(energies))
        
        for i in range(1, len(X_train_batches)):
            train_deq_batch(
                T_inference,
                X_train_batches[i],
                jax.nn.one_hot(y_train_batches[i], 10),
                model=model, optim_h=optim_h, optim_w=optim_w
            )
        
        # Eval with fixed inference
        correct = sum(eval_deq_batch(T_inference, X_test_batches[i], y_test_batches[i], 
                                      model=model, optim_h=optim_h)
                      for i in range(len(X_test_batches)))
        acc = correct / n_test * 100
        print(f"Epoch {epoch+1}: Test Acc = {acc:.2f}%")
    
    deq_results[name] = {'energies': np.array(first_batch_energies), 'accuracy': float(acc)}

NameError: name 'optimizer_configs' is not defined

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors = plt.cm.plasma(np.linspace(0.2, 0.9, n_epochs))

for idx, (name, data) in enumerate(deq_results.items()):
    ax = axes[idx]
    for epoch in range(n_epochs):
        ax.plot(data['energies'][epoch], color=colors[epoch], alpha=0.8)
    ax.set_xlabel('Inference Step t', fontsize=11)
    ax.set_ylabel('Energy E(t)', fontsize=11)
    ax.set_title(f'{name}\nFinal Acc: {data["accuracy"]:.1f}%', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

plt.suptitle('DEQ-PC: Energy → 0 ⟺ Fixed Point Found', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [9]:
# New Cell
print("\n" + "="*60)
print("DEQ-PC RESULTS")
print("="*60)
for name, data in deq_results.items():
    print(f"{name:<25} {data['accuracy']:>10.2f}%")


DEQ-PC RESULTS
